In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes safetensors sentencepiece --upgrade

In [ ]:
import math
import os
from functools import partial

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    GenerationConfig,
    LlamaForCausalLM,
    Trainer,
    TrainingArguments,
)

- LoRA: low-rank adaptation, rank selection, alpha scaling, target modules
- QLoRA: quantized base + LoRA adapters
- DoRA: weight-decomposed low-rank adaptation
- Adapter layers, prefix tuning, prompt tuning
When to use full fine-tuning vs PEFT
- Merging LoRA adapters: linear merge, TIES, DARE

# Introduction

![](https://cdn-uploads.huggingface.co/production/uploads/666b9ef5e6c60b6fc4156675/sIGOBaAZmS9QV7HfWaQKV.png)

---

Fine-tuning large language models is expensive — a 7B parameter model needs **~28GB** just in fp16 weights. **LoRA** solves this by **freezing the original weights** and injecting small, trainable low-rank matrices into specific layers (typically Q and V attention projections).


> The Problem: Full Fine-Tuning is Massive

In a standard transformer, the **query projection** in attention is a full weight matrix:
```
W_q shape: (4096, 4096) = 16,777,216 parameters  # for a 7B model like LLaMA-2
```

In full fine-tuning, **all 16.7M parameters** in this single layer need gradients, optimizer states, etc.

---

> The LoRA Solution: Low-Rank Decomposition

Instead of updating `W_q` directly, LoRA **freezes** it and learns two small matrices:
```
W_q: (4096, 4096)  ← FROZEN, no gradients

B:   (4096, 8)      ← trainable     (d × r)
A:   (8, 4096)      ← trainable     (r × d)
```

> **With `r = 8`:**
> - Original `W_q` parameters: `4096 × 4096` = **16,777,216**
> - LoRA parameters: `(4096 × 8) + (8 × 4096)` = **65,536**
> - Reduction: **99.6% fewer trainable parameters** for this layer

---

> The Math: Forward Pass
```python
# Standard (full fine-tuning):
output = x @ W_q                           # (batch, seq, 4096) @ (4096, 4096)

# LoRA (frozen W + trainable low-rank update):
output = x @ W_q + (x @ A @ B) * (α / r)  # frozen    + trainable update
#        ^^^^^^^^   ^^^^^^^^^^^^^
#        frozen      LoRA path
```

---

🎛️ Key Hyperparameters: `r` and `alpha`

| Parameter | What it controls | Typical values |
|-----------|-----------------|----------------|
| **`r` (rank)** | Size of the low-rank matrices. Lower = fewer params, higher = more expressive | 4, 8, 16 |
| **`lora_alpha`** | Scaling factor that controls how much LoRA influences the output | 16, 32 |
| **`lora_alpha / r`** | The **effective scaling**. This is what actually gets multiplied with the LoRA output | — |

> 🧮 **Example:** With `r=8` and `alpha=32`, the effective scale is `32/8 = 4`.
> This means the LoRA update is **multiplied by 4** before being added to the frozen weight output.

> 🤔 **Why not just use a learning rate?**
> The `alpha/r` scaling means you can **change the rank without retuning the learning rate**. If you double `r` from 8 → 16, the scaling automatically halves from `4 → 2`, keeping training stable.

> 📏 **How to pick `r`?**
> - `r=4` or `r=8` works well for most tasks — surprisingly small ranks capture enough information.
> - Higher `r` (16, 32) may help for complex tasks but adds more trainable parameters.
> - The QLoRA paper uses `r=8` with `alpha=32` as a good default.

---

> ⚡ **QLoRA** takes this further — loads the base model in **4-bit precision** (NF4 quantization), reducing memory from **~14GB → ~4GB** for a 7B model.

---

📌 What this notebook covers:

| Topic | Description |
|-------|-------------|
| **4-bit Quantization** | `BitsAndBytesConfig` with NF4 + double quantization |
| **LoRA Adapters** | Injected on Llama-2-7B via `peft` library |
| **Loss Masking** | Train only on response tokens, not the prompt |
| **From-Scratch LoRA (Wrapper)** | Subclassing attention with LoRA-enhanced Q/V |
| **From-Scratch LoRA (Partial)** | Wrapping `nn.Linear` with a parallel LoRA branch |

---

💬 Sample Training Prompt (LLaMA-2 chat format):
```
<s>[INST] <<SYS>> You are a helpful assistant. <</SYS>>
Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
What is the capital of France?

### Response: [/INST]
Paris is the capital of France.
```

# 1) Model Loading

⚙️ Model Loading with QLoRA 4-bit Quantization

---

### Key Concepts

> 📦 **Gradient Accumulation** — Simulates a larger batch size when GPU memory is limited. Instead of processing 64 samples at once, we process 16 at a time and accumulate gradients over 4 steps before updating weights.
>
> `effective_batch_size = micro_batch_size × gradient_accumulation_steps`

> 🖥️ **Distributed Data Parallel (DDP)** — When multiple GPUs are available, each GPU processes a portion of the data in parallel. The accumulation steps are divided by GPU count to keep the global batch size constant.

> 🧊 **QLoRA 4-bit Quantization** — The "Q" in QLoRA. Loads the base model in 4-bit precision, reducing memory from **~14GB (fp16) → ~4GB (4-bit)**. Three key settings:
>
> | Setting | What it does |
> |---------|-------------|
> | **NF4** | Quantization levels follow a normal distribution, matching how pretrained weights are distributed |
> | **Double Quantization** | Quantizes the quantization constants themselves, saving ~0.4 bits/param |
> | **bf16 Compute** | 4-bit weights are dequantized to bf16 on-the-fly during matrix multiplications |


In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    print(
        f"trainable model parameters: {trainable_model_params}\n"
        f"all model parameters: {all_model_params}"
    )
    return trainable_model_params

In [ ]:
model_id = "NousResearch/Llama-2-7b-hf"
max_length = 256
device_map = "auto"

# --- Gradient Accumulation ---
batch_size = 64
micro_batch_size = 16
gradient_accumulation_steps = batch_size // micro_batch_size  # 4 steps before weight update

# --- Distributed Data Parallel (DDP) ---
world_size = int(os.environ.get("WORLD_SIZE", 1))
ddp = world_size != 1

if ddp:
    device_map = {"": int(os.environ.get("LOCAL_RANK") or 0)}
    gradient_accumulation_steps = gradient_accumulation_steps // world_size

# --- QLoRA 4-bit Quantization Config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,  # quantize the quantization constants
    bnb_4bit_quant_type="nf4",  # normal-float-4, better than plain int4
    bnb_4bit_compute_dtype=torch.bfloat16,  # dequantize to bf16 for matmuls
)

# --- Load Model ---
model = LlamaForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map=device_map,
)
model.config.pretraining_tp = 1  # disable tensor parallelism (standard for QLoRA)

# Fallback: naive model parallelism if multiple GPUs but no DDP
if not ddp and torch.cuda.device_count() > 1:
    model.is_parallelizable = True
    model.model_parallel = True

# Before LoRA — all params are quantized/frozen, trainable count should be 0
ori_p = print_number_of_trainable_model_parameters(model)

🔧 **Preparing a Quantized Model for Training**

---

When using a **4-bit quantized model**, we need to prepare it before attaching LoRA adapters. The code below does this **manually** — the `peft` library provides `prepare_model_for_kbit_training()` that does the same thing in one line.

---

 1️⃣ Freeze all base model weights

Only LoRA adapters (added later) will be trained — the original model stays frozen.
```python
for param in model.parameters():
    param.requires_grad = False
```

---

 2️⃣ Cast small parameters to fp32

1D parameters like **LayerNorm** weights and biases are numerically sensitive — keeping them in fp16/4-bit can cause training instability.
```python
    if param.ndim == 1:
        param.data = param.data.to(torch.float32)
```

---

 3️⃣ Enable gradient checkpointing

Trades **compute for memory** — instead of storing all activations during the forward pass, it **recomputes** them during backward. Essential for fitting large models on limited GPU memory.
```python
model.gradient_checkpointing_enable()
```

---

 4️⃣ Enable input gradients

When the base model is frozen, gradients can get blocked. This ensures gradients still **flow through to the LoRA adapters**.
```python
model.enable_input_require_grads()
```

---

 5️⃣ Cast lm_head output to fp32

The loss function (cross-entropy) needs **fp32 precision** — this wrapper ensures the final output is cast before loss computation.
```python
class CastOutputToFloat(nn.Sequential):
    def forward(self, x):
        return super().forward(x).to(torch.float32)

model.lm_head = CastOutputToFloat(model.lm_head)
```

---

> ✅ **Shortcut:** You can replace this entire block with one line:
> ```python
> from peft import prepare_model_for_kbit_training
> model = prepare_model_for_kbit_training(model)
> ```

In [ ]:
from transformers import LlamaTokenizer
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# --- Tokenizer ---
tokenizer = LlamaTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# --- Prepare model for k-bit training ---
# Freezes params, casts small params (layernorm) to fp32, enables gradient checkpointing
model = prepare_model_for_kbit_training(model)

# --- LoRA Configuration (based on QLoRA paper) ---
# r: rank of low-rank matrices (lower = fewer params, higher = more expressive)
# lora_alpha: scaling factor (effective scale = alpha / r)
# target_modules: which attention projections get LoRA adapters
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
peft_p = print_number_of_trainable_model_parameters(model)
print(f"Before LoRA: {ori_p:,} | After LoRA: {peft_p:,} | {round(peft_p / ori_p * 100, 2)}%")

trainable model parameters: 4194304
 all model parameters: 3504607232 
# Trainable Parameter 
Before: 262410240 
After: 4194304 
Percentage: 1.6


# 2) Data Loading

In [ ]:
# --- Prompt Templates for LLaMA-2 chat format ---
prompt_template = {
    "prompt_input": (
        "<s>[INST] <<SYS>> You are a helpful assistant. <</SYS>> "
        "Below is an instruction that describes a task, paired with an input that provides further context. "
        "Write a response that appropriately completes the request."
        "\n\n### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response: [/INST]\n"
    ),
    "prompt_no_input": (
        "<s>[INST] <<SYS>> You are a helpful assistant. <</SYS>> "
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        "\n\n### Instruction:\n{instruction}\n\n### Response: [/INST]\n"
    ),
    "response_split": "### Response:",
}


def generate_prompt(instruction, input=None, label=None, prompt_template=prompt_template):
    """Build a prompt string from instruction + optional input, appending label if provided."""
    if input:
        res = prompt_template["prompt_input"].format(instruction=instruction, input=input)
    else:
        res = prompt_template["prompt_no_input"].format(instruction=instruction)
    if label:
        res = f"{res}{label}"
    return res

In [ ]:
def tokenize(tokenizer, prompt, max_length=max_length):
    """Tokenize a prompt and set labels = input_ids (for causal LM training)."""
    # Output keys: input_ids (seq_len,), attention_mask (seq_len,), labels (seq_len,)
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors=None,
    )
    result["labels"] = result["input_ids"].copy()
    return result


def generate_and_tokenize_prompt(data_point):
    """
    Tokenize full prompt+response, then mask the prompt portion in labels with -100
    so the model only computes loss on the response tokens.
    """
    full_prompt = generate_prompt(
        data_point["instruction"],
        data_point["context"],
        data_point["response"],
    )
    tokenized_full_prompt = tokenize(tokenizer, full_prompt)

    # Tokenize just the user prompt (without response) to find its length
    user_prompt = generate_prompt(data_point["instruction"], data_point["context"])
    tokenized_user_prompt = tokenize(tokenizer, user_prompt)
    user_prompt_len = len(tokenized_user_prompt["input_ids"])

    # Mask prompt tokens with -100 so loss is only computed on response tokens
    # labels: [-100, -100, ..., -100, response_token_1, response_token_2, ...]
    mask_token = [-100] * user_prompt_len
    tokenized_full_prompt["labels"] = (
        mask_token + tokenized_full_prompt["labels"][user_prompt_len:]
    )

    return tokenized_full_prompt

📦 Data Collators: `DataCollatorForSeq2Seq` vs `DataCollatorForLanguageModeling`

---

Data collators are responsible for **batching and preparing** tokenized examples before they're fed into the model. Different training objectives need different collators.

---

 1️⃣ `DataCollatorForSeq2Seq` — Used for Instruction Tuning / Causal LM

This collator is designed for tasks where we have **(prompt, response)** pairs and want the model to **only learn to generate the response**.

**Step 1: Pad sequences to equal length within the batch**

Each example may have different lengths. The collator pads them so they can form a batch tensor.
```python
# Before padding (different lengths):
input_ids = [[101, 202, 303],
             [101, 202, 303, 404, 505]]

# After padding (pad_token_id = 0):
input_ids = [[101, 202, 303,   0,   0],
             [101, 202, 303, 404, 505]]

# Labels padded with -100 (ignored by cross-entropy loss):
labels    = [[-100, -100, 303,  -100, -100],   # prompt tokens also masked
             [-100, -100, 303,   404,  505]]
#             ^^^^^^^^^^^                        ^^^^^^^^^^^^
#             prompt (masked)                    padding (masked)
```

**Step 2: Optional padding to multiple of 8**

GPU tensor cores work most efficiently with dimensions that are multiples of 8.
```python
# Sequence length = 5, pad_to_multiple_of = 8
# 8 - (5 % 8) = 3 extra pad tokens added

input_ids = [[101, 202, 303,   0,   0,   0,   0,   0],   # length 5 -> 8
             [101, 202, 303, 404, 505,   0,   0,   0]]
```

> 🔑 **Why don't we need to shift labels by `[1:]`?**
>
> HuggingFace causal LM models **internally handle the right-shift** in their loss computation:
> ```python
> # Inside LlamaForCausalLM.forward():
> shift_logits = logits[..., :-1, :]   # predictions for positions 0 to n-1
> shift_labels = labels[..., 1:]        # ground truth for positions 1 to n
>
> # Example:
> # input_ids = [A,  B,  C,  D]
> # logits     = [predict_B, predict_C, predict_D, predict_E]
> # After shift:
> # shift_logits = [predict_B, predict_C, predict_D]  # model's guess
> # shift_labels = [B,         C,         D]           # actual next token
> ```
> So we just pass `labels = input_ids` (with prompt tokens masked to `-100`) and the model takes care of alignment.

---

2️⃣ `DataCollatorForLanguageModeling` — Used for MLM (Masked Language Modeling)

This collator is designed for **BERT-style pretraining** where random tokens are corrupted and the model predicts the originals.

**Step 1: Save original tokens as labels**
```python
# Original input:
input_ids = [The, cat, sat, on, the, mat]
labels    = [The, cat, sat, on, the, mat]   # copy — this is the ground truth
```

**Step 2: Randomly select 15% of tokens to corrupt**
```python
# Suppose "cat" and "the" are selected (15% of 6 ≈ 1-2 tokens)
selected = [cat, the]
```

**Step 3: Apply the 80/10/10 corruption rule**
```python
# Of the selected tokens:
# 80% -> replace with [MASK]
# 10% -> replace with random token
# 10% -> keep unchanged

# "cat" -> [MASK]   (80% chance)
# "the" -> "banana"  (10% chance — random token)

input_ids = [The, [MASK], sat, on, banana, mat]   # corrupted input
labels    = [The,  cat,   sat, on,  the,   mat]   # original (ground truth)
#                  ^^^              ^^^
#                  model must predict these from context
```

> ⚠️ This is **NOT** for autoregressive / causal LM training. It's for **bidirectional** models like BERT and RoBERTa that can see both left and right context.

---

🔍 Side-by-Side Comparison

| Feature | `DataCollatorForSeq2Seq` | `DataCollatorForLanguageModeling` |
|---------|--------------------------|-----------------------------------|
| **Used for** | Instruction tuning, causal LM | BERT-style masked pretraining |
| **Model type** | GPT, LLaMA, Pythia (autoregressive) | BERT, RoBERTa (bidirectional) |
| **What gets masked** | **Prompt tokens** → `-100` in labels | **Random 15%** of tokens in input_ids |
| **Labels** | `labels = input_ids` (prompt = `-100`) | `labels = original input_ids` (before corruption) |
| **Input corruption** | None — input_ids stay intact | Yes — `[MASK]`, random, or unchanged |
| **Goal** | Learn to **generate responses** | Learn to **reconstruct masked tokens** |
| **Right-shift** | Handled internally by HF causal LM | Not needed (not autoregressive) |
| **Padding** | To max length + optional multiple of 8 | Standard padding |

In [ ]:
dataset = datasets.load_dataset("databricks/databricks-dolly-15k", split="train")
dataset = dataset.train_test_split(test_size=1000, shuffle=True, seed=42)

# --- Data collator: pads sequences to equal length within each batch ---
# Input: list of dicts with input_ids, attention_mask, labels
# Output: padded batch tensors (batch_num, seq_len)
data_collator = DataCollatorForSeq2Seq(
    tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
)

cols = ["instruction", "context", "response", "category"]
train_data = dataset["train"].shuffle().map(generate_and_tokenize_prompt, remove_columns=cols)
train_data = train_data.filter(lambda rec: len(rec["input_ids"]) < max_length)
val_data = dataset["test"].shuffle().map(generate_and_tokenize_prompt, remove_columns=cols)
val_data = val_data.filter(lambda rec: len(rec["input_ids"]) < max_length)

# --- Verify collator output ---
val_batch = data_collator(list(iter(val_data)))
n = 100
for k, v in val_batch.items():
    print(k, v[n])

print("\nDecoding input_ids\n", tokenizer.decode(val_batch["input_ids"][n]))
print("\nDecoding labels\n", tokenizer.decode([x for x in val_batch["labels"][n] if x > 0]))

# 3) Model Training

In [ ]:
trainer = Trainer(
    model=model,
    train_dataset=train_data,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=1000,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="./llama-7b-int4-dolly",
        optim="paged_adamw_8bit",  # 8-bit AdamW to save optimizer memory
    ),
    data_collator=data_collator,
)

# Disable KV cache during training (incompatible with gradient checkpointing)
model.config.use_cache = False
trainer.train()
model.save_pretrained("llama-7b-int4-dolly_1")

# 4) Generation

In [ ]:
# Reload base model + LoRA adapters for inference
model_id = "NousResearch/Llama-2-7b-hf"
peft_path = "llama-7b-int4-dolly_1"

# Load base model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    use_cache=False,
    device_map="auto",
)

# Load trained LoRA adapters onto the base model
model = PeftModel.from_pretrained(model, peft_path, torch_dtype=torch.float16)
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Switch to evaluation mode (disables dropout)
model.eval()

# --- Generation config: controls decoding strategy ---
generation_config = GenerationConfig(
    temperature=0.1,  # Low temperature = more deterministic
    top_p=0.75,       # Nucleus sampling threshold
    top_k=40,         # Top-k sampling
    num_beams=4,      # Beam search width
)

# --- Run inference ---
with torch.no_grad():
    prompt = "<s>[INST] <<SYS>> You are a helpful assistant. <</SYS>> Write me a poem about Singapore. [/INST]"

    # Tokenize prompt
    # Output: input_ids (1, seq_len), attention_mask (1, seq_len)
    inputs = tokenizer(prompt, return_tensors="pt")

    # Generate tokens
    # Input: input_ids (1, seq_len) -> Output: sequences (1, seq_len + num_new_tokens)
    generation_output = model.generate(
        input_ids=inputs.input_ids,
        generation_config=generation_config,
        return_dict_in_generate=True,
        output_scores=True,
        max_new_tokens=64,
    )

    # Decode generated token IDs back to text
    # Input: (num_tokens,) -> Output: string
    print("\nAnswer: ", tokenizer.decode(generation_output.sequences[0]))

# 5) Lora From Scratch

![](https://miro.medium.com/v2/resize:fit:720/format:webp/0*RVttR2w2DpWNpxdV.png)

## 5.1 Wrapper Approach

### 5.1.1 Base Multi-Head Self-Attention (for reference)

In [ ]:
class RobertaSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_attention_heads
        self.head_size = config.hidden_size // config.num_attention_heads
        self.all_head_size = config.hidden_size

        # Q, K, V linear projections
        # Input: (batch_num, seq_len, hidden_dim) -> Output: (batch_num, seq_len, hidden_dim)
        self.query = nn.Linear(config.hidden_size, config.hidden_size)
        self.key = nn.Linear(config.hidden_size, config.hidden_size)
        self.value = nn.Linear(config.hidden_size, config.hidden_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

    def reshape_for_heads(self, x):
        """Reshape for multi-head attention.
        Input: (batch_num, seq_len, hidden_dim) -> Output: (batch_num, num_heads, seq_len, head_size)
        """
        batch_num, seq_len, _ = x.size()
        return x.view(batch_num, seq_len, self.num_heads, self.head_size).permute(0, 2, 1, 3)

    def forward(self, hidden_states, attention_mask=None):
        """
        Standard scaled dot-product multi-head self-attention.
        Input: hidden_states (batch_num, seq_len, hidden_dim)
        Input: attention_mask (batch_num, 1, 1, seq_len) - additive mask
        Output: (batch_num, seq_len, hidden_dim)
        """
        # Project and reshape into multiple heads
        # (batch_num, seq_len, hidden_dim) -> (batch_num, num_heads, seq_len, head_size)
        Q = self.reshape_for_heads(self.query(hidden_states))
        K = self.reshape_for_heads(self.key(hidden_states))
        V = self.reshape_for_heads(self.value(hidden_states))

        # Scaled dot-product attention scores
        # (batch_num, num_heads, seq_len, head_size) @ (batch_num, num_heads, head_size, seq_len)
        # -> (batch_num, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.head_size)

        # Apply additive mask (padding positions get -inf -> 0 after softmax)
        if attention_mask is not None:
            scores = scores + attention_mask

        # Softmax -> attention weights + dropout
        # (batch_num, num_heads, seq_len, seq_len)
        attn_weights = self.dropout(F.softmax(scores, dim=-1))

        # Weighted sum of values
        # (batch_num, num_heads, seq_len, seq_len) @ (batch_num, num_heads, seq_len, head_size)
        # -> (batch_num, num_heads, seq_len, head_size)
        context = torch.matmul(attn_weights, V)

        # Concatenate heads back together
        # (batch_num, num_heads, seq_len, head_size) -> (batch_num, seq_len, hidden_dim)
        batch_num, _, seq_len, _ = context.size()
        context = context.permute(0, 2, 1, 3).contiguous().view(batch_num, seq_len, -1)

        return context

### 5.1.2 LoRA-enhanced Self-Attention

In [ ]:
class LoraRobertaSelfAttention(RobertaSelfAttention):
    """
    Adds LoRA low-rank matrices to Q and V projections.
    W_new = W_frozen + (B @ A) * (alpha / r)
    Only B and A are trainable; the original W is frozen.
    """

    def __init__(self, r=8, alpha=1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        d = self.all_head_size

        # LoRA matrices: B initialized to zeros (so LoRA starts as identity)
        # A initialized with random values
        # Input: (hidden_dim, r) and (r, hidden_dim)
        self.lora_query_matrix_B = nn.Parameter(torch.zeros(d, r))
        self.lora_query_matrix_A = nn.Parameter(torch.randn(r, d))
        self.lora_value_matrix_B = nn.Parameter(torch.zeros(d, r))
        self.lora_value_matrix_A = nn.Parameter(torch.randn(r, d))

        # Scaling factor: reduces need to retune learning rate for different r values
        self.scale = alpha / r

    def lora_query(self, x):
        """Apply frozen W_q + trainable LoRA to query projection.
        Input: (batch_num, seq_len, hidden_dim) -> Output: (batch_num, seq_len, hidden_dim)
        """
        # LoRA weight: (hidden_dim, r) @ (r, hidden_dim)
        # Output: (hidden_dim, hidden_dim)
        lora_query_weights = torch.matmul(self.lora_query_matrix_B, self.lora_query_matrix_A)

        # F.linear(input, weight) computes: input @ weight.T
        # Input: frozen_output + lora_output * scale
        # Output: (batch, seq_len, hidden_dim)
        return self.query(x) + F.linear(x, lora_query_weights) * self.scale

    def lora_value(self, x):
        """Apply frozen W_v + trainable LoRA to value projection.
        Input: (batch_num, seq_len, hidden_dim) -> Output: (batch_num, seq_len, hidden_dim)
        """
        # LoRA weight: (hidden_dim, r) @ (r, hidden_dim)
        # Output(hidden_dim, hidden_dim)
        lora_value_weights = torch.matmul(self.lora_value_matrix_B, self.lora_value_matrix_A)

        # F.linear(input, weight) computes: input @ weight.T
        # Input: frozen_output + lora_output * scale
        # Output: (batch, seq_len, hidden_dim)
        return self.value(x) + F.linear(x, lora_value_weights) * self.scale

    def forward(self, hidden_states, attention_mask=None):
        """Override forward to use LoRA-enhanced Q and V projections.
        Input: (batch_num, seq_len, hidden_dim) -> Output: (batch_num, seq_len, hidden_dim)
        """
        # Q and V use LoRA; K stays unchanged
        # (batch_num, seq_len, hidden_dim) -> (batch_num, num_heads, seq_len, head_size)
        Q = self.reshape_for_heads(self.lora_query(hidden_states))
        K = self.reshape_for_heads(self.key(hidden_states))
        V = self.reshape_for_heads(self.lora_value(hidden_states))

        # Scaled dot-product attention (same as base class)
        # (batch_num, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.head_size)
        if attention_mask is not None:
            scores = scores + attention_mask
        attn_weights = self.dropout(F.softmax(scores, dim=-1))

        # (batch_num, num_heads, seq_len, head_size)
        context = torch.matmul(attn_weights, V)

        # Concatenate heads: (batch_num, seq_len, hidden_dim)
        batch_num, _, seq_len, _ = context.size()
        context = context.permute(0, 2, 1, 3).contiguous().view(batch_num, seq_len, -1)
        return context

### 5.1.3 LoRA Wrapper for full RoBERTa model

In [ ]:
class LoraWrapperRoberta(nn.Module):
    """
    Wraps a pre-trained RoBERTa model with LoRA adapters.
    Replaces attention layers with LoRA-enhanced versions and freezes all
    parameters except LoRA matrices, biases, and layer norms.
    """

    def __init__(
        self,
        task_type,
        num_classes=None,
        dropout_rate=0.1,
        model_id="roberta-large",
        lora_rank=8,
        train_biases=True,
        train_embedding=False,
        train_layer_norms=True,
    ):
        super().__init__()
        from transformers import RobertaModel, RobertaTokenizer

        self.model_id = model_id
        self.lora_rank = lora_rank
        self.train_biases = train_biases
        self.train_embeddings = train_embedding
        self.train_layer_norms = train_layer_norms

        self.tokenizer = RobertaTokenizer.from_pretrained(model_id)
        self.model = RobertaModel.from_pretrained(model_id)
        self.model_config = self.model.config

        model_dim = self.model_config.hidden_size

        # Classification head
        # Input: (batch_num, seq_len, model_dim) -> Output: (batch_num, seq_len, model_dim)
        self.finetune_head_norm = nn.LayerNorm(model_dim)
        self.finetune_head_dropout = nn.Dropout(dropout_rate)
        # Input: (batch_num, model_dim) -> Output: (batch_num, num_classes)
        self.finetune_head_classifier = nn.Linear(model_dim, num_classes)

        # Replace attention layers with LoRA versions and freeze base params
        self.replace_multihead_attention_recursion(self.model)
        self.freeze_parameters_except_lora_and_bias()

    def replace_multihead_attention_recursion(self, model):
        """Recursively replace RobertaSelfAttention with LoraRobertaSelfAttention."""
        for name, module in model.named_children():
            if isinstance(module, RobertaSelfAttention):
                new_layer = LoraRobertaSelfAttention(r=self.lora_rank, config=self.model_config)
                new_layer.load_state_dict(module.state_dict(), strict=False)
                setattr(model, name, new_layer)
            else:
                self.replace_multihead_attention_recursion(module)

    def freeze_parameters_except_lora_and_bias(self):
        """Freeze all parameters except LoRA matrices, classification head, and configured layers."""
        for name, param in self.model.named_parameters():
            is_trainable = (
                "lora_" in name
                or "finetune_head_" in name
                or (self.train_biases and "bias" in name)
                or (self.train_embeddings and "embeddings" in name)
                or (self.train_layer_norms and "LayerNorm" in name)
            )
            param.requires_grad = is_trainable

## 5.2 Partial Function Approach (simpler, modular)

In [ ]:
lora_r = 8
lora_alpha = 16
lora_dropout = 0.05
lora_query = True
lora_key = False
lora_value = True
lora_projection = False
lora_mlp = False
lora_head = False

In [ ]:
class LoRALayer(nn.Module):
    """A single low-rank adaptation layer: x -> alpha * (x @ A @ B)"""

    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        # A: (in_dim, rank) initialized with scaled random normal
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        # B: (rank, out_dim) initialized to zeros (LoRA starts as no-op)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # Input: (batch_num, seq_len, in_dim)
        # x @ A: (batch_num, seq_len, rank)
        # x @ A @ B: (batch_num, seq_len, out_dim)
        return self.alpha * (x @ self.A @ self.B)


class LinearWithLoRA(nn.Module):
    """Wraps an existing nn.Linear with a parallel LoRA branch.
    Output = frozen_linear(x) + lora(x)
    """

    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear  # Frozen original layer
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        # Input: (batch_num, seq_len, in_dim) -> Output: (batch_num, seq_len, out_dim)
        return self.linear(x) + self.lora(x)

In [ ]:
# --- Load a pre-trained model for demonstration ---
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

# Create a partial function with fixed LoRA hyperparameters for clean injection
assign_lora = partial(LinearWithLoRA, rank=lora_r, alpha=lora_alpha)

# --- Inject LoRA layers into each transformer block ---
for layer in model.distilbert.transformer.layer:
    if lora_query:
        # Input: (batch_num, seq_len, model_dim) -> Output: (batch_num, seq_len, model_dim)
        layer.attention.q_lin = assign_lora(layer.attention.q_lin)
    if lora_key:
        # Input: (batch_num, seq_len, model_dim) -> Output: (batch_num, seq_len, model_dim)
        layer.attention.k_lin = assign_lora(layer.attention.k_lin)
    if lora_value:
        # Input: (batch_num, seq_len, model_dim) -> Output: (batch_num, seq_len, model_dim)
        layer.attention.v_lin = assign_lora(layer.attention.v_lin)
    if lora_projection:
        # Input: (batch_num, seq_len, model_dim) -> Output: (batch_num, seq_len, model_dim)
        layer.attention.out_lin = assign_lora(layer.attention.out_lin)
    if lora_mlp:
        # Input: (batch_num, seq_len, model_dim) -> Output: (batch_num, seq_len, hidden_dim)
        layer.ffn.lin1 = assign_lora(layer.ffn.lin1)
        # Input: (batch_num, seq_len, hidden_dim) -> Output: (batch_num, seq_len, model_dim)
        layer.ffn.lin2 = assign_lora(layer.ffn.lin2)

# Optionally apply LoRA to the classification head
if lora_head:
    # Input: (batch_num, model_dim) -> Output: (batch_num, model_dim)
    model.pre_classifier = assign_lora(model.pre_classifier)
    # Input: (batch_num, model_dim) -> Output: (batch_num, num_classes)
    model.classifier = assign_lora(model.classifier)